#### Dataset
- Name: Titanic Dataset
- Task: Binary Classification
- Source: Kaggle
- Reference: Link: https://www.kaggle.com/c/titanic


<h1> Naive Bayes 

Naive Bayes Theory

Naive Bayes operates based on Bayes' theorem:

$$ P(y|X)= \frac{P(X|y)P(y)} {P(X)} $$

where:

Posterior Probability

The posterior probability of the class:

$$ P(y|X) $$

Meaning:

Given the features, what is the probability that the sample belongs to the class?

### Prior Probability

Class prior probability:

$$ P(y) $$

For example:

In the Titanic dataset:

$$ P(Survived=0) $$

and

$$ P(Survived=1) $$

...before seeing the features.

### Likelihood

The probability of observing features given the class:

$$ P(X|y) $$

For example:

The probability that:

A female passenger with a high fare belongs to the "Survivor" class.

Why is it called "Naive"?

Because it relies on a simplifying assumption:

The features are independent of one another.

That is, it assumes:

$$ P(X|y) = P(x_1|y) P(x_2|y) ... P(x_n|y) $$

For example, it assumes:

Age is independent of Fare.

This is not always true in the real world.

Yet, despite this simplifying assumption, it performs well on many problems.

Gaussian Naive Bayes

Since our features are numerical, for each feature we assume:

It follows a Gaussian distribution:

$$ X \sim N(\mu,\sigma^2) $$

Probability function:

$$ P(x)= \frac{1}{\sqrt{2\pi\sigma^2}} e^{-\frac{(x-\mu)^2}{2\sigma^2}} $$

For each class, the model learns:

Mean

Variance

...for each feature.

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from src.knn import KNearestNeighbors
from sklearn.neighbors import KNeighborsClassifier

from sklearn.naive_bayes import GaussianNB

<h3> Step 1: Dataset

In [9]:
df = pd.read_csv(
    "../../../datasets/Titanic-Dataset/processed/titanic_feature_engineered.csv"
)

df.head()

,Survived,Age,Fare,FamilySize,FarePerPerson,SibSp,Parch,Sex,Embarked,FamilyCategory,Title,Deck
0,0,22.0,7.2500,2,3.62500,1,0,male,S,Small,Mr,Unknown
1,1,38.0,71.2833,2,35.64165,1,0,female,C,Small,Mrs,C
2,1,26.0,7.9250,1,7.92500,0,0,female,S,Alone,Miss,Unknown
3,1,35.0,53.1000,2,26.55000,1,0,female,S,Small,Mrs,C
4,0,35.0,8.0500,1,8.05000,0,0,male,S,Alone,Mr,Unknown


In [10]:
X = df.drop(
    "Survived",
    axis=1
)

y = df["Survived"]

In [11]:
numeric_features = X.select_dtypes(
    include=[
        "int64",
        "float64"
    ]
).columns.tolist()


categorical_features = X.select_dtypes(
    include=[
        "object"
    ]
).columns.tolist()

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [13]:
numeric_transformer = Pipeline(
    steps=[
        
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),

        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[

        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[

        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [14]:
X_train_processed = preprocessor.fit_transform(
    X_train
)

X_test_processed = preprocessor.transform(
    X_test
)

<h3> Step 2: Model Building

In [15]:
nb_model = GaussianNB()

<h3> Step 3: Training

In [16]:
nb_model.fit(
    X_train_processed,
    y_train
)

GaussianNB()

<h3> Step 4: Prediction

In [17]:
y_pred_nb = nb_model.predict(
    X_test_processed
)

<h3> Step 5: Evaluation

In [18]:
nb_accuracy = accuracy_score(
    y_test,
    y_pred_nb
)

nb_precision = precision_score(
    y_test,
    y_pred_nb
)

nb_recall = recall_score(
    y_test,
    y_pred_nb
)

nb_f1 = f1_score(
    y_test,
    y_pred_nb
)

print(
    "Accuracy:",
    nb_accuracy
)

print(
    "Precision:",
    nb_precision
)

print(
    "Recall:",
    nb_recall
)

print(
    "F1-score:",
    nb_f1
)

Accuracy: 0.770949720670391
Precision: 0.6555555555555556
Recall: 0.855072463768116
F1-score: 0.7421383647798742


<h3> Step 6: Classification Report

In [19]:
print(
    classification_report(
        y_test,
        y_pred_nb
    )
)

              precision    recall  f1-score   support

           0       0.89      0.72      0.79       110
           1       0.66      0.86      0.74        69

    accuracy                           0.77       179
   macro avg       0.77      0.79      0.77       179
weighted avg       0.80      0.77      0.77       179



<h3> Step 7: Confusion Matrix

In [20]:
cm_nb = confusion_matrix(
    y_test,
    y_pred_nb
)

cm_nb

array([[79, 31],
       [10, 59]], dtype=int64)

<h3> Step 8: Probability Prediction

In [21]:
y_prob_nb = nb_model.predict_proba(
    X_test_processed
)

<h3> Step 9: Compare With Previous Models

| Model               | Accuracy | Precision(1) | Recall(1) |    F1 |
| ------------------- | -------: | -----------: | --------: | ----: |
| Logistic Regression |    0.827 |        0.806 |     0.725 | 0.763 |
| KNN (k=25)          |    0.821 |        0.794 |     0.725 | 0.758 |
| Naive Bayes         |    0.771 |        0.656 |     0.855 | 0.742 |


#### Comparison of the philosophies of the three models


| Model               | Approach       | رفتار                   |
| ------------------- | -------------- | ----------------------- |
| Logistic Regression | Discriminative | Learning the decision boundary |
| KNN                 | Distance Based | Similarity to previous examples |
| Naive Bayes         | Probabilistic  | Calculating class probabilities |


### Naive Bayes Results

Gaussian Naive Bayes achieved an accuracy of 77.1% on the Titanic test set.

Although its accuracy was lower compared with Logistic Regression and KNN, the model achieved a high recall for the Survivor class (85.5%).

This means that Naive Bayes successfully identified most actual survivors but produced more false positive predictions.

The behavior is related to the probabilistic nature of Naive Bayes and its assumption that features are conditionally independent given the class label.

The results demonstrate that different classification algorithms optimize different aspects of prediction performance.